In [18]:
import altair as alt
import numpy as np
import pandas as pd

# Causal Thompson Sampling

In [20]:
payout_df = pd.DataFrame({
    "drunk": [0, 0, 0, 0, 1, 1, 1, 1],
    "blink": [0, 1, 0, 1, 0, 1, 0, 1],
    "machine": [0, 0, 1, 1, 0, 0, 1, 1],
    "payout": [0.1, 0.5, 0.5, 0.1, 0.4, 0.2, 0.2, 0.4]
})

payout_df

,drunk,blink,machine,payout
0,0,0,0,0.1
1,0,1,0,0.5
2,0,0,1,0.5
3,0,1,1,0.1
4,1,0,0,0.4
5,1,1,0,0.2
6,1,0,1,0.2
7,1,1,1,0.4


In [343]:
alt.Chart(payout_df, width=150, height=150).mark_rect().encode(
    column="drunk:O",
    x="blink:O",
    y="machine:O",
    color="payout"
)

alt.Chart(...)

In [533]:
class ThompsonSampling:
    success = [1, 1]
    failure = [1, 1]

    def __call__(self, x):
        a1 = np.random.beta(self.success[0], self.failure[0])
        a2 = np.random.beta(self.success[1], self.failure[1])

        return np.argmax([a1, a2])

    def update(self, x, a, r):
        self.success[a] += r
        self.failure[a] += 1 - r

class ContextualThompsonSampling:
    success = np.ones([2, 2, 2])
    failure = np.ones([2, 2, 2])

    def __call__(self, x):
        s1 = self.success[x[0], x[1], 0]
        s2 = self.success[x[0], x[1], 1]
        f1 = self.failure[x[0], x[1], 0]
        f2 = self.failure[x[0], x[1], 1]

        a1 = np.random.beta(s1, f1)
        a2 = np.random.beta(s2, f2)
        return np.argmax([a1, a2])

    def update(self, x, a, r):
        self.success[x[0], x[1], a] += r
        self.failure[x[0], x[1], a] += 1 - r

class CausalThompsonSampling:
    success = np.ones([2, 2])
    failure = np.ones([2, 2])
    intuition = None
    
    def __call__(self, x):
        intuition = int(np.logical_xor(x[0], x[1]))
        counter_intuition = 1 - intuition
        self.intuition = intuition
        
        Q1 = self.success[intuition, counter_intuition] / (self.success[intuition, counter_intuition] + self.failure[intuition, counter_intuition])
        Q2 = self.success[intuition, intuition] / (self.success[intuition, intuition] + self.failure[intuition, intuition])
        bias = 1 - np.abs(Q1 - Q2)
        weights = [1, 1]

        if Q1 > Q2:
            weights[intuition] = bias
        else:
            weights[counter_intuition] = bias
        
        a1 = np.random.beta(self.success[intuition, intuition], self.failure[intuition, intuition]) * weights[0]
        a2 = np.random.beta(self.success[counter_intuition, counter_intuition], self.failure[counter_intuition, counter_intuition]) * weights[1]
        return np.argmax([a1, a2])

    def update(self, x, a, r):
        self.success[self.intuition, a] += r
        self.failure[self.intuition, a] += 1 - r
        self.intuition = None

class UniformPolicy:
    def __call__(self, x):
        return np.random.randint(2)

    def update(self, x, a, r):
        pass

class DrunkPolicy:
    def __call__(self, x):
        return int(np.logical_xor(x[0], x[1]))

    def update(self, x, a, r):
        pass

In [534]:
def get_reward(context, action):
    context2payout = payout_df.set_index(["drunk", "blink", "machine"]).payout.to_dict()
    payout = context2payout[context[0], context[1], action]

    return np.random.binomial(1, payout), payout

def get_regret(context, action):
    context2payout = payout_df.set_index(["drunk", "blink", "machine"]).payout.to_dict()
    payout = context2payout[context[0], context[1], action]
    counterfactual_payout = context2payout[context[0], context[1], 1 - action]
    return max(counterfactual_payout - payout, 0)

def simulate(policy, name, trials=10_000):
    rewards = [0, 0]
    regret = 0
    results = []

    for t in range(trials):
        context = np.random.binomial(1, [0.5, 0.5], (2,))
        action = policy(context)
    
        reward, payout = get_reward(context, action)
        rewards[action] += reward
        regret += get_regret(context, action)

        policy.update(context, action, reward)
    
        results.append({
            "policy": name,
            "t": t,
            "regret": regret,
            "reward": np.sum(rewards),
        })

    return pd.DataFrame(results)


result_df = pd.concat([
    simulate(DrunkPolicy(), "DrunkPolicy"),
    simulate(ThompsonSampling(), "ThompsonSampling"),
    simulate(UniformPolicy(), "UniformPolicy"),
    simulate(CausalThompsonSampling(), "CausalThompsonSampling"),
])

In [539]:
alt.data_transformers.disable_max_rows()

alt.Chart(result_df, title="Greedy Casino Example", width=500, height=350).mark_line().encode(
    x=alt.X("t", title="Trials"),
    y=alt.Y("reward", title="Cummulative Reward"),
    color="policy"
) | alt.Chart(result_df, title="Greedy Casino Example", width=500, height=350).mark_line().encode(
    x=alt.X("t", title="Trials"),
    y=alt.Y("regret:Q", title="Cummulative Regret"),
    color="policy",
    tooltip=["regret"]
)

alt.HConcatChart(...)